In [3]:
import pandas as pd 
import numpy as np 
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [61]:
fp = ("US-Ne1", "US-Ne2", "US-Ne3", "US-Var")
month_names = ('Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec')
comp_vars = ("SIF_ET", "Ensemble", "geeSEBAL", "PT-JPL", "SSEBop", "SIMS", "eeMETRIC", "DisALEXI", "Climatology")
results = []
month_occ = []
for fn in fp: 
    df = pd.read_csv(fn+"_clim.csv", delimiter = ',', parse_dates=["Date"])
    df["Date"] = pd.to_datetime(df["Date"])
    df.set_index("Date", inplace=True)
    for i, name in enumerate(month_names):
        month_df = df[df.index.month == i + 1]
        for var in comp_vars:
            subset = month_df[["observed_ET", var]].dropna()
            if subset.empty:
                continue

            y = subset[var].values.reshape(-1, 1)
            obs = subset["observed_ET"].values

            model = LinearRegression().fit(y, obs)
            y_pred = model.predict(y)
            r2 = r2_score(obs, y_pred)

            y = y.reshape(-1, )
            y_pred = y_pred.reshape(-1, )

            RMSE = np.sqrt(mean_squared_error(obs, y))
            MAE = mean_absolute_error(obs, y)
            MBE = (sum(y-obs))/len(obs)

            SRMSE = RMSE/(np.std(obs))

            results.append({
                "Site": fn,
                "month": name,
                "Variable": var,
                "R2": round(r2, 3),
                "RMSE": round(RMSE, 3),
                "SRMSE": round(SRMSE, 3),
                "MAE": round(MAE,3),
                "MBE": MBE
                })
        month_occ.append({
            "Site": fn, 
            "month": name,
            "# of occurrences": len(subset)
            })
            

vars = ["Ensemble", "SIF_ET", "Climatology",'eeMETRIC', 'geeSEBAL', 'DisALEXI', 'SSEBop', 'PT-JPL', 'SIMS']
results_df = pd.DataFrame(results)
month_occ = pd.DataFrame(month_occ)
display(results_df)
#results_df.to_csv("loc.csv")
#display(results_df.describe())

,Site,month,Variable,R2,RMSE,SRMSE,MAE,MBE
0,US-Ne1,Jan,SIF_ET,0.206,13.812,3.374,13.269,13.268698
1,US-Ne1,Jan,Ensemble,0.294,6.839,1.751,5.906,-5.863729
2,US-Ne1,Jan,geeSEBAL,0.039,9.658,2.473,8.224,-5.588729
3,US-Ne1,Jan,PT-JPL,0.185,7.134,1.827,6.094,-4.569979
4,US-Ne1,Jan,SSEBop,0.004,9.601,2.459,8.443,-7.701229
...,...,...,...,...,...,...,...,...
415,US-Var,Dec,PT-JPL,0.001,16.497,6.274,15.423,15.423181
416,US-Var,Dec,SSEBop,0.025,7.121,2.708,5.974,-3.326819
417,US-Var,Dec,eeMETRIC,0.061,8.736,3.322,6.735,-0.820569
418,US-Var,Dec,DisALEXI,0.013,6.829,2.597,5.741,-5.483069


In [62]:
table = month_occ.pivot(index="Site", columns="month", values="# of occurrences")
table = table.reindex(month_names, axis=1)
display(table)

month,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
Site,,,,,,,,,,,,
US-Ne1,18,18,18,18,18,19,19,19,19,19,19,19
US-Ne2,18,18,18,18,18,19,19,19,19,19,19,19
US-Ne3,18,18,18,18,18,18,19,19,19,19,19,19
US-Var,20,20,21,21,21,21,21,21,20,20,20,20


In [85]:
for var in vars:
    table = results_df[results_df["Variable"] == var].pivot(index="Site", columns="month", values="R2")
    table = table.reindex(month_names, axis=1)
    table = round(table,2)
    #table = table.style.format("{:.2}").background_gradient(cmap='Blues', vmin=0, vmax=0.8)
    #if var == "SIF_ET":
        #table.set_caption(var[:3] + " " + var[4:] + " R²")
    #else:
        #table.set_caption(var+" R²")
    #table.to_latex(var+"_r2.txt", convert_css = True)
    display(table)
    

month,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
Site,,,,,,,,,,,,
US-Ne1,0.29,0.23,0.54,0.12,0.41,0.19,0.74,0.47,0.15,0.37,0.03,0.01
US-Ne2,0.63,0.01,0.32,0.13,0.52,0.58,0.75,0.10,0.11,0.06,0.01,0.11
US-Ne3,0.22,0.12,0.23,0.00,0.09,0.14,0.46,0.07,0.00,0.01,0.04,0.38
US-Var,0.02,0.11,0.14,0.47,0.73,0.44,0.03,0.03,0.01,0.12,0.20,0.04


month,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
Site,,,,,,,,,,,,
US-Ne1,0.21,0.32,0.01,0.01,0.00,0.26,0.36,0.08,0.16,0.07,0.05,0.10
US-Ne2,0.25,0.10,0.01,0.04,0.06,0.16,0.17,0.17,0.10,0.03,0.01,0.02
US-Ne3,0.03,0.11,0.01,0.00,0.00,0.12,0.02,0.00,0.04,0.20,0.15,0.01
US-Var,0.06,0.26,0.25,0.42,0.85,0.61,0.00,0.12,0.03,0.59,0.77,0.17


month,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
Site,,,,,,,,,,,,
US-Ne1,0.0,0.0,0.0,0.0,0.0,-0.0,0.0,0.0,0.0,0.0,0.0,0.0
US-Ne2,0.0,-0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.0,0.0,0.0,0.0
US-Ne3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.0,0.0,0.0,0.0,0.0
US-Var,0.0,-0.0,0.0,-0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.0,0.0


month,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
Site,,,,,,,,,,,,
US-Ne1,0.09,0.08,0.11,0.11,0.27,0.24,0.51,0.20,0.13,0.29,0.06,0.00
US-Ne2,0.06,0.04,0.01,0.06,0.29,0.47,0.56,0.02,0.07,0.15,0.04,0.04
US-Ne3,0.09,0.32,0.04,0.04,0.00,0.02,0.17,0.01,0.00,0.00,0.00,0.10
US-Var,0.00,0.18,0.34,0.57,0.78,0.57,0.00,0.11,0.00,0.49,0.14,0.06


month,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
Site,,,,,,,,,,,,
US-Ne1,0.04,0.08,0.08,0.08,0.56,0.18,0.48,0.05,0.11,0.07,0.10,0.01
US-Ne2,0.41,0.00,0.17,0.18,0.55,0.73,0.62,0.00,0.13,0.07,0.01,0.00
US-Ne3,0.02,0.05,0.17,0.01,0.14,0.36,0.36,0.08,0.03,0.00,0.25,0.12
US-Var,0.08,0.01,0.01,0.24,0.49,0.39,0.01,0.00,0.18,0.02,0.03,0.03


month,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
Site,,,,,,,,,,,,
US-Ne1,0.02,0.34,0.56,0.13,0.37,0.07,0.28,0.49,0.16,0.11,0.00,0.02
US-Ne2,0.10,0.00,0.11,0.04,0.40,0.54,0.62,0.50,0.06,0.01,0.03,0.01
US-Ne3,0.04,0.00,0.07,0.02,0.06,0.07,0.33,0.16,0.04,0.06,0.00,0.21
US-Var,0.18,0.07,0.17,0.48,0.69,0.33,0.00,0.00,0.01,0.13,0.54,0.01


month,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
Site,,,,,,,,,,,,
US-Ne1,0.00,0.27,0.03,0.01,0.25,0.19,0.64,0.32,0.12,0.40,0.02,0.13
US-Ne2,0.05,0.05,0.11,0.00,0.46,0.52,0.66,0.09,0.04,0.10,0.00,0.02
US-Ne3,0.00,0.01,0.00,0.10,0.34,0.15,0.33,0.10,0.01,0.03,0.04,0.06
US-Var,0.18,0.45,0.34,0.60,0.59,0.13,0.04,0.05,0.00,0.02,0.46,0.02


month,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
Site,,,,,,,,,,,,
US-Ne1,0.18,0.33,0.54,0.18,0.35,0.08,0.55,0.01,0.09,0.03,0.01,0.02
US-Ne2,0.43,0.02,0.52,0.00,0.61,0.43,0.39,0.00,0.03,0.02,0.02,0.07
US-Ne3,0.21,0.05,0.34,0.01,0.32,0.18,0.31,0.00,0.00,0.02,0.16,0.17
US-Var,0.18,0.23,0.54,0.02,0.52,0.84,0.28,0.15,0.13,0.32,0.50,0.00


month,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
Site,,,,,,,,,,,,
US-Ne1,0.10,0.11,0.04,0.33,0.53,0.14,0.60,0.29,0.13,0.34,0.12,0.13
US-Ne2,0.18,0.24,0.01,0.62,0.65,0.41,0.56,0.08,0.09,0.38,0.37,0.42
US-Ne3,0.29,0.07,0.00,0.44,0.62,0.16,0.24,0.01,0.12,0.48,0.40,0.27


In [64]:
for var in vars:
    table = results_df[results_df["Variable"] == var].pivot(index="Site", columns="month", values="RMSE")
    table = table.reindex(month_names, axis=1)
    table = table.style.format("{:.1f}").background_gradient(cmap='Reds', vmin=0, vmax=50)
    if var == "SIF_ET":
        table.set_caption(var[:3] + " RMSE")
    else:
        table.set_caption(var+" RMSE")
    table.to_latex(var+"_RMSE.txt", convert_css = True)
    display(table) 

month,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
Site,,,,,,,,,,,,
US-Ne1,6.8,16.1,18.3,20.0,24.0,33.6,24.6,20.5,21.0,15.4,8.6,7.4
US-Ne2,6.5,15.4,19.1,20.8,21.8,27.6,23.9,24.6,22.7,15.8,9.2,7.0
US-Ne3,5.1,10.0,13.5,16.7,19.5,23.6,16.5,19.8,24.5,13.2,5.9,4.0
US-Var,5.6,7.9,14.1,13.0,28.2,47.3,48.9,43.9,24.4,13.2,5.5,5.3


month,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
Site,,,,,,,,,,,,
US-Ne1,13.8,8.6,15.4,17.0,20.5,40.5,74.7,45.1,22.7,11.6,10.6,14.6
US-Ne2,15.2,8.5,9.3,9.7,15.9,30.7,63.8,45.8,23.8,11.5,13.8,15.9
US-Ne3,16.2,8.7,13.9,13.4,11.8,15.7,38.4,33.4,21.5,14.1,16.1,17.7
US-Var,24.2,17.8,8.5,8.9,15.8,37.8,43.8,41.4,35.5,30.3,26.8,23.9


month,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
Site,,,,,,,,,,,,
US-Ne1,4.1,8.8,11.1,13.0,16.2,23.7,23.1,17.5,19.6,7.3,6.3,4.8
US-Ne2,3.9,7.2,7.6,8.9,16.1,28.7,30.4,17.6,19.8,4.4,4.9,3.9
US-Ne3,5.6,4.4,13.1,12.8,11.0,16.6,22.2,22.9,18.7,5.8,3.7,3.0
US-Var,2.6,4.4,8.5,11.4,20.9,14.1,2.3,2.8,2.0,4.0,4.6,2.6


month,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
Site,,,,,,,,,,,,
US-Ne1,8.1,19.6,26.1,26.5,40.1,37.6,29.5,21.1,34.9,32.2,14.4,9.9
US-Ne2,8.2,17.7,27.0,31.3,41.6,35.3,31.3,25.0,37.5,28.6,13.5,8.6
US-Ne3,5.9,12.3,21.3,24.4,34.6,29.2,31.2,22.4,37.8,23.7,8.5,6.0
US-Var,8.6,9.5,14.4,21.2,30.0,26.2,31.3,35.1,18.5,12.4,6.8,8.7


month,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
Site,,,,,,,,,,,,
US-Ne1,9.7,16.3,26.8,24.9,33.7,42.7,34.3,32.7,25.2,14.1,10.6,8.9
US-Ne2,7.5,16.6,22.9,24.9,38.4,36.2,36.6,34.2,25.1,12.9,9.9,8.6
US-Ne3,7.5,11.8,19.9,25.1,29.3,30.5,27.9,32.6,26.1,10.2,6.9,7.7
US-Var,8.0,10.6,18.4,19.7,43.6,71.3,80.0,77.4,48.1,20.6,10.5,4.7


month,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
Site,,,,,,,,,,,,
US-Ne1,7.4,10.8,13.8,21.8,28.4,42.4,19.9,13.3,20.3,13.2,11.2,6.9
US-Ne2,7.7,15.1,18.6,21.8,26.0,31.7,22.8,13.3,25.4,17.6,12.8,7.1
US-Ne3,7.2,10.1,15.9,22.9,22.6,29.5,25.8,18.3,25.8,15.5,9.0,4.4
US-Var,7.3,11.0,20.0,14.4,23.1,50.0,56.7,47.0,20.9,9.0,4.5,6.8


month,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
Site,,,,,,,,,,,,
US-Ne1,9.6,20.5,31.7,32.9,32.8,38.8,25.5,17.5,28.5,19.8,8.6,9.5
US-Ne2,9.0,20.0,30.2,30.9,27.2,32.1,32.0,21.4,32.8,23.9,11.9,9.3
US-Ne3,7.6,14.8,23.3,22.1,25.7,24.8,29.2,18.7,31.6,20.1,9.8,6.7
US-Var,7.5,11.3,19.3,10.8,20.5,43.7,51.8,47.5,25.0,12.1,5.0,7.1


month,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
Site,,,,,,,,,,,,
US-Ne1,7.1,14.6,11.7,21.4,13.3,33.5,45.1,41.4,20.7,14.7,9.6,7.0
US-Ne2,5.3,14.3,8.7,15.3,10.7,25.9,38.3,44.1,22.5,13.4,7.7,6.4
US-Ne3,5.2,9.3,8.5,13.5,14.4,25.4,18.3,35.8,24.9,12.8,6.8,5.2
US-Var,16.0,14.5,9.3,23.9,59.7,53.8,42.8,34.6,27.3,25.7,21.4,16.5


month,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
Site,,,,,,,,,,,,
US-Ne1,6.4,13.9,16.9,19.0,30.3,34.4,31.1,25.0,23.5,25.3,8.2,6.4
US-Ne2,5.4,11.3,14.9,22.8,33.9,36.8,26.6,27.9,26.0,25.7,7.1,5.0
US-Ne3,4.1,8.4,15.0,26.9,40.5,44.2,23.4,22.2,30.2,27.6,9.4,5.2


In [65]:
for var in vars:
    table = results_df[results_df["Variable"] == var].pivot(index="Site", columns="month", values="MAE")
    table = table.reindex(month_names, axis=1)
    table = table.style.format("{:.1f}").background_gradient(cmap='Reds', vmin=0, vmax=50)
    if var == "SIF_ET":
        table.set_caption(var[:3] + " MAE")
    else:
        table.set_caption(var+" MAE")
    table.to_latex(var+"_MAE.txt", convert_css = True)
    display(table)

month,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
Site,,,,,,,,,,,,
US-Ne1,5.9,14.2,16.5,16.5,21.5,25.1,22.5,17.3,16.0,13.1,6.8,6.3
US-Ne2,6.1,12.6,17.9,16.6,19.3,19.4,21.5,20.1,19.0,13.6,7.3,5.9
US-Ne3,4.3,8.4,11.2,14.7,17.1,19.1,13.3,16.6,21.1,10.9,4.8,3.5
US-Var,4.6,6.0,12.5,9.9,23.5,45.0,46.6,41.4,22.4,11.7,4.6,3.8


month,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
Site,,,,,,,,,,,,
US-Ne1,13.3,7.3,11.7,14.0,16.7,34.8,67.6,37.1,18.3,10.6,9.0,13.8
US-Ne2,14.7,7.5,7.6,7.3,12.3,25.6,53.5,37.0,17.3,10.4,12.9,15.3
US-Ne3,15.6,7.7,9.1,8.6,9.5,13.4,34.5,25.2,17.7,13.2,15.8,17.4
US-Var,23.9,17.1,5.8,7.4,11.0,36.0,43.7,41.3,35.5,30.1,26.7,23.8


month,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
Site,,,,,,,,,,,,
US-Ne1,3.5,7.4,8.6,9.7,13.6,18.7,16.7,14.9,15.2,5.6,5.3,4.2
US-Ne2,2.9,5.4,6.1,6.4,13.3,22.1,23.4,15.1,13.2,3.8,4.0,3.3
US-Ne3,3.6,3.7,8.8,7.2,8.8,13.5,17.5,15.7,15.5,4.5,2.9,2.5
US-Var,1.7,3.7,6.9,8.2,16.1,10.6,1.7,1.5,1.4,3.3,3.5,2.0


month,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
Site,,,,,,,,,,,,
US-Ne1,7.0,17.2,22.9,24.2,35.0,29.3,26.1,15.6,28.0,27.6,11.7,8.8
US-Ne2,6.9,15.4,24.6,28.2,37.2,26.1,29.0,19.9,30.4,23.0,10.6,7.4
US-Ne3,5.0,11.4,18.1,21.5,31.6,24.8,23.6,17.9,31.8,19.6,6.7,4.7
US-Var,6.9,7.4,12.0,18.1,26.1,23.8,27.8,32.0,16.2,10.5,5.2,6.7


month,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
Site,,,,,,,,,,,,
US-Ne1,8.2,13.4,22.1,20.4,28.3,34.5,28.5,27.2,18.7,11.3,8.6,8.2
US-Ne2,6.5,13.8,20.4,21.0,34.1,30.2,27.5,29.6,19.1,10.3,7.9,7.3
US-Ne3,6.5,10.3,15.9,22.1,25.1,26.7,22.2,27.0,20.6,8.3,5.9,6.6
US-Var,6.6,8.2,16.6,14.8,35.2,66.4,75.2,72.6,44.8,17.6,9.2,4.2


month,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
Site,,,,,,,,,,,,
US-Ne1,5.9,9.8,12.2,19.0,25.3,33.5,16.5,11.2,17.3,10.4,9.5,5.7
US-Ne2,6.6,12.3,16.5,18.2,23.1,22.6,19.4,9.5,20.3,14.9,11.4,6.1
US-Ne3,5.9,8.4,12.4,17.3,21.0,24.2,22.6,15.2,22.9,13.4,8.0,3.8
US-Var,6.2,9.3,17.2,10.3,19.4,46.2,52.9,43.2,17.3,6.7,3.8,5.7


month,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
Site,,,,,,,,,,,,
US-Ne1,8.4,18.8,29.3,29.5,30.1,32.9,22.6,14.2,21.8,15.8,7.3,8.3
US-Ne2,8.4,17.6,28.4,28.3,23.9,25.1,28.4,18.3,27.1,20.6,9.7,8.2
US-Ne3,6.8,13.7,21.3,20.2,23.8,19.5,22.4,15.7,25.8,17.6,8.3,5.8
US-Var,6.5,9.6,17.8,9.1,15.9,38.1,47.6,42.7,21.6,9.1,4.1,6.0


month,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
Site,,,,,,,,,,,,
US-Ne1,6.1,13.2,9.1,17.4,10.3,25.8,42.1,36.0,16.0,11.8,7.3,5.9
US-Ne2,4.6,12.2,7.3,12.4,9.1,16.5,31.7,38.1,15.3,9.7,6.2,5.1
US-Ne3,3.9,8.2,6.1,9.8,11.5,22.2,14.7,29.6,21.4,11.0,5.4,4.2
US-Var,15.4,12.7,7.8,19.4,57.6,53.3,42.3,34.2,26.4,24.1,20.4,15.4


month,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
Site,,,,,,,,,,,,
US-Ne1,5.4,11.0,13.1,15.6,27.9,28.3,29.3,21.8,17.3,23.0,6.6,5.0
US-Ne2,4.3,8.6,12.2,21.5,32.2,32.6,22.7,23.8,19.4,23.7,5.6,4.4
US-Ne3,3.5,6.4,11.7,25.0,39.0,41.4,15.2,18.1,26.1,26.0,7.5,3.9


In [66]:
for var in vars:
    table = results_df[results_df["Variable"] == var].pivot(index="Site", columns="month", values="MBE")
    table = table.reindex(month_names, axis=1)
    table = table.style.format("{:.1f}").background_gradient(cmap='seismic', vmin=-45, vmax=45)
    #table = table.style.format("{:.3f}").background_gradient(cmap='seismic', vmin=-45, vmax=45)
    if var == "SIF_ET":
        table.set_caption(var[:3] + " MBE")
    else:
        table.set_caption(var+" MBE")
    table.to_latex(var+"_MBE.txt", convert_css = True)
    display(table)

month,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
Site,,,,,,,,,,,,
US-Ne1,-5.9,-13.9,-16.5,-13.9,-18.0,-13.3,-20.9,-15.4,5.6,12.7,-4.3,-5.6
US-Ne2,-6.1,-12.4,-17.9,-16.0,-14.7,-14.7,-16.3,-17.6,2.4,6.3,-3.6,-5.8
US-Ne3,-3.7,-8.1,-10.8,-4.9,-10.9,-6.0,-0.4,-9.4,9.2,4.2,-1.5,-2.9
US-Var,-2.2,-4.5,-7.8,3.8,22.3,45.0,46.6,41.4,22.4,11.4,2.3,-2.2


month,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
Site,,,,,,,,,,,,
US-Ne1,13.3,2.5,-10.1,-8.6,-12.0,-34.5,-67.6,-34.5,-13.1,9.1,8.6,13.8
US-Ne2,14.7,4.9,-4.1,-1.2,-2.6,-15.0,-52.0,-34.2,-13.1,10.4,12.9,15.3
US-Ne3,15.2,7.7,-3.6,-0.4,2.1,-1.8,-26.3,-11.2,6.6,13.1,15.8,17.4
US-Var,23.9,17.1,-1.1,-1.6,7.7,36.0,43.7,41.3,35.5,30.1,26.7,23.8


month,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
Site,,,,,,,,,,,,
US-Ne1,0.0,0.0,0.0,0.0,0.0,-0.0,-0.0,0.0,0.0,0.0,0.0,-0.0
US-Ne2,-0.0,-0.0,-0.0,0.0,0.0,-0.0,-0.0,-0.0,0.0,-0.0,0.0,0.0
US-Ne3,-0.0,0.0,-0.0,-0.0,0.0,0.0,-0.0,0.0,0.0,-0.0,0.0,-0.0
US-Var,0.0,0.0,-0.0,0.0,0.0,0.0,0.0,0.0,-0.0,0.0,-0.0,0.0


month,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
Site,,,,,,,,,,,,
US-Ne1,-7.0,-17.2,-22.9,-17.8,-32.4,-12.4,-16.0,-4.8,23.9,26.9,4.9,-4.2
US-Ne2,-6.5,-15.4,-24.6,-24.6,-33.0,-16.3,-11.1,-6.7,22.3,15.4,0.6,-5.6
US-Ne3,-4.5,-10.8,-17.4,-11.5,-23.1,-10.8,6.0,0.1,24.9,12.7,-0.5,-3.9
US-Var,-2.4,-1.1,3.0,13.5,22.7,23.5,27.8,32.0,16.2,10.5,2.4,-0.8


month,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
Site,,,,,,,,,,,,
US-Ne1,-5.6,-11.9,-9.0,-8.4,-19.7,-21.6,-28.1,-26.3,-6.1,4.5,-7.2,-6.4
US-Ne2,-5.5,-9.9,-14.1,-13.7,-16.7,-27.1,-27.1,-27.4,-9.5,-3.6,-6.1,-6.3
US-Ne3,-5.1,-7.2,-7.9,-2.7,-8.4,-20.0,-19.2,-24.3,-3.4,-0.3,-2.7,-3.9
US-Var,-0.9,-6.7,-9.6,9.3,30.2,66.4,75.2,72.6,44.8,17.2,6.0,-1.4


month,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
Site,,,,,,,,,,,,
US-Ne1,-0.4,-5.0,-11.3,-16.9,-25.3,-19.2,1.3,0.7,-4.5,0.5,-5.8,-2.5
US-Ne2,-3.1,-8.1,-16.1,-17.3,-20.1,-20.3,-1.1,-3.5,-10.6,-7.0,-8.5,-4.6
US-Ne3,-0.3,-3.0,-4.6,-5.2,-15.6,-8.0,5.9,1.3,-3.6,-7.3,-6.3,-1.7
US-Var,-3.0,-8.0,-17.2,-8.3,17.6,46.2,52.9,43.2,17.3,4.6,-1.6,-5.5


month,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
Site,,,,,,,,,,,,
US-Ne1,-7.7,-18.8,-29.3,-29.0,-26.6,-17.5,-8.9,-3.6,9.2,15.0,-2.1,-6.6
US-Ne2,-7.6,-17.6,-28.4,-28.3,-21.0,-17.2,-8.1,-5.7,6.3,7.1,-3.6,-6.8
US-Ne3,-5.1,-13.7,-21.3,-17.5,-23.8,-10.9,4.5,0.6,9.9,0.7,-4.2,-4.0
US-Var,-5.7,-9.6,-16.2,-3.9,12.8,38.1,47.6,42.7,21.5,7.3,-2.3,-3.3


month,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
Site,,,,,,,,,,,,
US-Ne1,-4.6,-12.4,-7.9,-4.3,-4.2,-9.4,-42.1,-36.0,1.7,9.5,-4.0,-4.5
US-Ne2,-3.1,-10.8,-4.4,-2.0,1.3,-5.1,-30.0,-38.1,-0.9,8.6,-1.0,-3.6
US-Ne3,-1.0,-6.2,-1.2,4.9,7.3,8.6,-9.9,-28.3,8.7,9.6,4.0,-0.6
US-Var,15.4,12.7,6.4,19.1,57.6,53.3,42.3,34.2,26.4,24.1,20.4,15.4


month,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
Site,,,,,,,,,,,,
US-Ne1,-3.8,-9.7,-7.3,12.9,27.9,17.6,-25.5,-19.4,13.7,22.9,-0.9,-2.7
US-Ne2,-3.0,-8.2,-2.0,18.9,32.2,23.5,-15.2,-20.7,14.3,23.3,2.2,-2.2
US-Ne3,-1.0,-3.7,2.8,22.8,39.0,36.9,7.6,-7.0,22.2,26.0,6.0,1.0


In [4]:
fp = ("US-Ne1", "US-Ne2", "US-Ne3", "US-Var")
season_names = ('DJF', 'MAM', 'JJA', 'SON')

comp_vars = ("SIF_ET", "Ensemble", "geeSEBAL", "PT-JPL", "SSEBop", "SIMS", "eeMETRIC", "DisALEXI", "Climatology")
season_results = []
month_occ = []
for fn in fp: 
    df = pd.read_csv(fn+"_clim.csv", delimiter = ',', parse_dates=["Date"])
    df["Date"] = pd.to_datetime(df["Date"])
    df.set_index("Date", inplace=True)
    season_filters = {
    "All Months": df,
    "DJF": df[df.index.month.isin([12, 1, 2])],
    "MAM": df[df.index.month.isin([3, 4, 5])],
    "JJA": df[df.index.month.isin([6, 7, 8])],
    "SON": df[df.index.month.isin([9, 10, 11])],
        }
    season_names = list(season_filters.keys())
    for i, name in enumerate(season_names):
        season_df = season_filters[name]
        for var in comp_vars:
            subset = season_df[["observed_ET", var]].dropna()
            if subset.empty:
                continue

            y = subset[var].values.reshape(-1, 1)
            obs = subset["observed_ET"].values

            model = LinearRegression().fit(y, obs)
            y_pred = model.predict(y)
            r2 = r2_score(obs, y_pred)

            y = y.reshape(-1, )
            y_pred = y_pred.reshape(-1, )

            RMSE = np.sqrt(mean_squared_error(obs, y))
            MAE = mean_absolute_error(obs, y)
            MBE = (sum(y-obs))/len(obs)

            SRMSE = RMSE/(np.std(obs))

            season_results.append({
                "Site": fn,
                "season": name,
                "Variable": var,
                "R2": round(r2, 3),
                "RMSE": round(RMSE, 3),
                "SRMSE": round(SRMSE, 3),
                "MAE": round(MAE,3),
                "MBE": MBE
                })
            

vars = ["Ensemble", "SIF_ET", "Climatology",'eeMETRIC', 'geeSEBAL', 'DisALEXI', 'SSEBop', 'PT-JPL', 'SIMS']
season_names = ["DJF", "MAM", "JJA", "SON"]
season_df = pd.DataFrame(season_results)
display(season_df)
#results_df.to_csv("loc2.csv")
#display(results_df.describe())

,Site,season,Variable,R2,RMSE,SRMSE,MAE,MBE
0,US-Ne1,All Months,SIF_ET,0.878,31.130,0.501,21.373,-11.272479
1,US-Ne1,All Months,Ensemble,0.923,19.546,0.312,15.142,-8.868181
2,US-Ne1,All Months,geeSEBAL,0.869,25.734,0.411,19.131,-12.122861
3,US-Ne1,All Months,PT-JPL,0.893,23.851,0.381,16.917,-9.878526
4,US-Ne1,All Months,SSEBop,0.880,24.971,0.398,19.821,-9.987885
...,...,...,...,...,...,...,...,...
170,US-Var,SON,PT-JPL,0.251,24.864,4.893,23.573,23.572522
171,US-Var,SON,SSEBop,0.050,16.136,3.175,11.453,8.609256
172,US-Var,SON,eeMETRIC,0.011,13.362,2.630,10.510,9.527624
173,US-Var,SON,DisALEXI,0.008,13.267,2.611,9.163,6.590889


In [5]:
columns = ("R2","RMSE","MAE", "MBE", "SRMSE")
dropped = ("season", "Variable")
for var in vars:
    table = season_df[season_df["Variable"] == var]
    table = table[table["season"]== "All Months"]
    table = table.drop("season", axis = 1)
    table = table.drop("Variable", axis = 1)
    table = table.set_index("Site")
    table = table.reindex(columns, axis=1)
    table = table.style.format({
        "R2": "{:.2f}",
        "RMSE": "{:.1f}",
        "MAE": "{:.1f}",
        "MBE":"{:.1f}"
    })
    if var == "SIF_ET":
        table.set_caption(var[:3] + " " + var[4:] + " All months Stats")
    else:
        table.set_caption(var+" All months stats")
    #table.to_latex(var+"_all_stats.txt", convert_css = True)
    display(table)

,R2,RMSE,MAE,MBE,SRMSE
Site,,,,,
US-Ne1,0.92,19.5,15.1,-8.9,0.312000
US-Ne2,0.92,19.3,15.1,-9.5,0.321000
US-Ne3,0.92,16.0,12.2,-3.7,0.302000
US-Var,0.52,26.9,19.3,14.8,0.919000


,R2,RMSE,MAE,MBE,SRMSE
Site,,,,,
US-Ne1,0.88,31.1,21.4,-11.3,0.501000
US-Ne2,0.87,27.6,18.6,-5.5,0.464000
US-Ne3,0.87,20.4,15.8,2.9,0.394000
US-Var,0.82,28.7,25.2,23.5,0.977000


,R2,RMSE,MAE,MBE,SRMSE
Site,,,,,
US-Ne1,0.94,14.7,10.3,0.0,0.236000
US-Ne2,0.93,15.8,10.0,0.0,0.267000
US-Ne3,0.93,13.6,8.7,0.0,0.263000
US-Var,0.91,8.9,5.1,0.0,0.303000


,R2,RMSE,MAE,MBE,SRMSE
Site,,,,,
US-Ne1,0.84,27.1,21.2,-6.0,0.432000
US-Ne2,0.83,27.9,21.8,-8.3,0.463000
US-Ne3,0.84,24.1,18.2,-2.9,0.454000
US-Var,0.74,20.9,16.1,12.2,0.715000


,R2,RMSE,MAE,MBE,SRMSE
Site,,,,,
US-Ne1,0.87,25.7,19.1,-12.1,0.411000
US-Ne2,0.87,25.6,19.1,-14.0,0.426000
US-Ne3,0.85,22.1,16.6,-8.9,0.417000
US-Var,0.21,43.9,30.9,25.1,1.501000


,R2,RMSE,MAE,MBE,SRMSE
Site,,,,,
US-Ne1,0.92,19.9,14.7,-7.2,0.318000
US-Ne2,0.92,19.9,15.2,-10.0,0.331000
US-Ne3,0.89,19.1,14.8,-4.0,0.361000
US-Var,0.38,28.7,19.9,11.4,0.981000


,R2,RMSE,MAE,MBE,SRMSE
Site,,,,,
US-Ne1,0.88,25.0,19.8,-10.0,0.398000
US-Ne2,0.88,25.2,20.5,-10.5,0.419000
US-Ne3,0.89,21.2,16.8,-6.8,0.400000
US-Var,0.41,27.0,19.0,10.6,0.925000


,R2,RMSE,MAE,MBE,SRMSE
Site,,,,,
US-Ne1,0.89,23.9,16.9,-9.9,0.381000
US-Ne2,0.90,21.8,14.2,-7.4,0.363000
US-Ne3,0.89,17.9,12.6,-0.5,0.338000
US-Var,0.70,32.7,27.5,27.3,1.119000


,R2,RMSE,MAE,MBE,SRMSE
Site,,,,,
US-Ne1,0.88,22.2,17.2,2.1,0.355000
US-Ne2,0.86,23.2,17.9,5.6,0.386000
US-Ne3,0.85,25.1,18.9,12.9,0.474000


In [8]:
for var in vars:
    table = season_df[season_df["Variable"] == var].pivot(index="Site", columns="season", values="SRMSE")
    table = table.reindex(season_names, axis=1)
    table = table.style.format("{:.1f}").background_gradient(cmap='Reds', vmin= 0, vmax=5)
    if var == "SIF_ET":
        table.set_caption(var[:3] + " " + var[4:] + " MBE")
    else:
        table.set_caption(var+" MBE")
    #table.to_latex(var+"_season_MBE.txt", convert_css = True)
    display(table)

season,DJF,MAM,JJA,SON
Site,,,,
US-Ne1,1.4,1.1,0.7,0.5
US-Ne2,1.5,1.2,0.6,0.5
US-Ne3,1.5,1.3,0.6,0.6
US-Var,0.8,1.0,4.0,3.2


season,DJF,MAM,JJA,SON
Site,,,,
US-Ne1,1.7,0.9,1.6,0.5
US-Ne2,2.1,0.7,1.2,0.5
US-Ne3,2.7,0.8,0.9,0.6
US-Var,2.6,0.7,3.7,6.2


season,DJF,MAM,JJA,SON
Site,,,,
US-Ne1,0.8,0.7,0.6,0.4
US-Ne2,0.8,0.6,0.7,0.4
US-Ne3,0.8,0.8,0.6,0.4
US-Var,0.4,0.8,0.8,0.7


season,DJF,MAM,JJA,SON
Site,,,,
US-Ne1,1.7,1.6,0.8,0.9
US-Ne2,1.8,1.9,0.8,0.8
US-Ne3,1.8,2.1,0.8,0.9
US-Var,1.1,1.2,2.6,2.6


season,DJF,MAM,JJA,SON
Site,,,,
US-Ne1,1.5,1.5,1.0,0.6
US-Ne2,1.7,1.7,0.9,0.5
US-Ne3,2.0,1.9,0.9,0.6
US-Var,1.0,1.6,6.4,6.0


season,DJF,MAM,JJA,SON
Site,,,,
US-Ne1,1.1,1.1,0.8,0.5
US-Ne2,1.5,1.3,0.6,0.6
US-Ne3,1.6,1.6,0.7,0.6
US-Var,1.0,1.0,4.3,2.6


season,DJF,MAM,JJA,SON
Site,,,,
US-Ne1,1.8,1.7,0.8,0.7
US-Ne2,2.0,1.7,0.7,0.7
US-Ne3,2.2,1.8,0.7,0.8
US-Var,1.1,0.9,4.0,3.2


season,DJF,MAM,JJA,SON
Site,,,,
US-Ne1,1.3,0.8,1.1,0.5
US-Ne2,1.4,0.7,0.9,0.5
US-Ne3,1.5,0.9,0.8,0.6
US-Var,1.9,2.0,3.8,4.9


season,DJF,MAM,JJA,SON
Site,,,,
US-Ne1,1.2,1.2,0.8,0.7
US-Ne2,1.1,1.4,0.7,0.6
US-Ne3,1.3,2.2,0.9,0.8


In [68]:
for var in vars:
    table = season_df[season_df["Variable"] == var].pivot(index="Site", columns="season", values="MBE")
    table = table.reindex(season_names, axis=1)
    table = table.style.format("{:.1f}").background_gradient(cmap='seismic', vmin= -35, vmax=35)
    if var == "SIF_ET":
        table.set_caption(var[:3] + " " + var[4:] + " MBE")
    else:
        table.set_caption(var+" MBE")
    table.to_latex(var+"_season_MBE.txt", convert_css = True)
    display(table)

season,DJF,MAM,JJA,SON
Site,,,,
US-Ne1,-8.3,-16.2,-16.6,4.7
US-Ne2,-7.9,-16.2,-16.2,1.7
US-Ne3,-4.8,-8.9,-5.3,3.9
US-Var,-3.0,6.1,44.4,11.9


season,DJF,MAM,JJA,SON
Site,,,,
US-Ne1,9.9,-10.2,-45.6,1.6
US-Ne2,11.7,-2.6,-33.7,3.4
US-Ne3,13.5,-0.6,-13.3,11.8
US-Var,21.6,1.7,40.3,30.8


season,DJF,MAM,JJA,SON
Site,,,,
US-Ne1,-0.0,0.0,-0.0,0.0
US-Ne2,-0.0,0.0,-0.0,0.0
US-Ne3,-0.0,-0.0,0.0,0.0
US-Var,0.0,0.0,0.0,-0.0


season,DJF,MAM,JJA,SON
Site,,,,
US-Ne1,-9.2,-24.3,-11.1,18.6
US-Ne2,-8.9,-27.4,-11.5,12.8
US-Ne3,-6.3,-17.3,-1.6,12.4
US-Var,-1.5,13.1,27.7,9.5


season,DJF,MAM,JJA,SON
Site,,,,
US-Ne1,-7.8,-12.3,-25.4,-3.0
US-Ne2,-7.1,-14.8,-27.2,-6.4
US-Ne3,-5.4,-6.3,-21.2,-2.2
US-Var,-3.0,10.0,71.4,22.3


season,DJF,MAM,JJA,SON
Site,,,,
US-Ne1,-2.6,-17.7,-5.6,-3.3
US-Ne2,-5.2,-17.8,-8.4,-8.7
US-Ne3,-1.7,-8.5,-0.3,-5.7
US-Var,-5.5,-2.7,47.5,6.6


season,DJF,MAM,JJA,SON
Site,,,,
US-Ne1,-10.8,-28.3,-10.0,7.4
US-Ne2,-10.4,-26.0,-10.4,3.3
US-Ne3,-7.5,-20.9,-1.9,2.1
US-Var,-6.3,-2.4,42.8,8.6


season,DJF,MAM,JJA,SON
Site,,,,
US-Ne1,-7.0,-5.5,-29.4,2.4
US-Ne2,-5.6,-1.7,-24.1,2.2
US-Ne3,-2.5,3.7,-10.2,7.4
US-Var,14.5,27.7,43.5,23.6


season,DJF,MAM,JJA,SON
Site,,,,
US-Ne1,-5.3,10.8,-9.4,11.9
US-Ne2,-4.3,16.4,-3.8,13.3
US-Ne3,-1.1,21.5,12.1,18.1


In [70]:
for var in vars:
    table = season_df[season_df["Variable"] == var].pivot(index="Site", columns="season", values="MAE")
    table = table.reindex(season_names, axis=1)
    table = table.style.format("{:.1f}").background_gradient(cmap='Reds', vmin=0, vmax=45)
    if var == "SIF_ET":
        table.set_caption(var[:3] + " " + var[4:] + " MAE")
    else:
        table.set_caption(var+" MAE")
    table.to_latex(var+"_season_MAE.txt", convert_css = True)
    display(table)

season,DJF,MAM,JJA,SON
Site,,,,
US-Ne1,8.6,18.1,21.6,12.0
US-Ne2,8.0,17.9,20.3,13.3
US-Ne3,5.3,14.3,16.3,12.3
US-Var,4.8,15.3,44.4,12.7


season,DJF,MAM,JJA,SON
Site,,,,
US-Ne1,11.5,14.1,46.5,12.6
US-Ne2,12.6,9.0,38.7,13.5
US-Ne3,13.6,9.1,24.6,15.6
US-Var,21.6,8.1,40.3,30.8


season,DJF,MAM,JJA,SON
Site,,,,
US-Ne1,5.0,10.6,16.8,8.7
US-Ne2,3.8,8.6,20.2,7.0
US-Ne3,3.2,8.3,15.6,7.6
US-Var,2.5,10.4,4.6,2.8


season,DJF,MAM,JJA,SON
Site,,,,
US-Ne1,10.8,27.3,23.7,22.5
US-Ne2,9.7,30.0,25.1,21.3
US-Ne3,7.0,23.7,22.0,19.3
US-Var,7.0,18.7,27.8,10.5


season,DJF,MAM,JJA,SON
Site,,,,
US-Ne1,9.8,23.6,30.0,12.9
US-Ne2,9.0,25.1,29.1,12.4
US-Ne3,7.8,21.1,25.3,11.6
US-Var,6.4,22.2,71.4,23.6


season,DJF,MAM,JJA,SON
Site,,,,
US-Ne1,7.1,18.7,20.3,12.4
US-Ne2,8.1,19.2,17.3,15.5
US-Ne3,6.0,16.9,20.6,14.8
US-Var,7.1,15.7,47.5,9.2


season,DJF,MAM,JJA,SON
Site,,,,
US-Ne1,11.6,29.6,23.2,14.9
US-Ne2,11.1,26.9,24.0,19.1
US-Ne3,8.7,21.8,19.1,17.3
US-Var,7.4,14.2,42.8,11.5


season,DJF,MAM,JJA,SON
Site,,,,
US-Ne1,8.2,12.2,34.8,11.7
US-Ne2,7.1,9.7,28.6,10.4
US-Ne3,5.4,9.1,22.3,12.6
US-Var,14.5,28.3,43.5,23.6


season,DJF,MAM,JJA,SON
Site,,,,
US-Ne1,7.0,18.8,26.5,15.7
US-Ne2,5.6,22.0,26.4,16.2
US-Ne3,4.6,25.2,24.8,19.8
